In [0]:
import os
import json

is_job = False
try:
    raw = dbutils.jobs.taskValues.get("join_customers_and_orders_data", "metadata", None)
    print("Notebook is running in Job mode")
    print(f"raw ---> {raw}")

    # get data from the json output of the job ingest_customers_data
    meta = json.loads(raw)
    catalog          = meta.get("catalog")
    schema           = meta.get("schema")
    table_name       = meta.get("table_name")
    gold_table_name = meta.get("gold_table_name")
    is_job = True
except:
    is_job = False
    

In [0]:
if is_job:
    spark.sql(f"""
            CREATE OR REPLACE TABLE {catalog}.{schema}.{gold_table_name} AS
                SELECT
                    customer_id,
                    first_name,
                    last_name,
                    phone,
                    order_id,
                    -- Convert string to DATE
                    TO_DATE(order_date, 'yyyy-MM-dd') AS order_date,
                    product,
                    category,
                    --CAST Quantity to integer
                    CAST(quantity AS INTEGER) AS quantity,
                    -- Convert string to DOUBLE
                    CAST(order_amount AS DOUBLE) AS order_amount,
                    state
                FROM {catalog}.{schema}.{table_name};
            """)
else:
    print(f"Notebook is running in interactive mode skipping data cleaning process!")